In [ ]:
from pathlib import Path
import automated_llm_probes as alp

TARGET_N = 450
LOCK20 = [
    "claude-haiku-4.5", "claude-opus-4.5", "claude-opus-4.7", "claude-opus-5",
    "claude-sonnet-4.5", "gpt-3.5-turbo", "gpt-4-turbo", "gpt-4o", "gpt-4o-mini",
    "gpt-5.4", "gpt-5.6-sol", "grok-4.2", "grok-4.3", "grok-4.5", "grok-4.6",
    "grok-build-0.1", "llama-3.1-8b", "llama-3.2-3b", "llama-4-maverick", "llama-4-scout"]

models = [m for m in alp.ready_models() if m["name"] in LOCK20]
print("N", TARGET_N, [m["name"] for m in models])

for m in models:
    have = sum(1 for p in Path("dat", m["name"]).rglob("*.pickle") if p.is_file())
    print(f"  {m['name']:28s} {have:4d}/{TARGET_N}  {'ok' if have >= TARGET_N else f'+{TARGET_N - have}'}")
    if have < TARGET_N:
        alp.collect("DAT", models=[m], n_per_model=TARGET_N)

N 450 ['grok-4.2', 'grok-4.3', 'grok-4.5', 'grok-build-0.1', 'grok-4.6', 'gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4o', 'gpt-5.4', 'gpt-5.6-sol', 'gpt-4o-mini', 'gpt-4-turbo', 'llama-4-scout', 'llama-4-maverick', 'llama-3.2-3b', 'llama-3.1-8b', 'claude-sonnet-4.5', 'claude-haiku-4.5', 'claude-opus-4.5', 'claude-opus-4.7', 'claude-opus-5']
  grok-4.2                        0/450  +450
  grok-4.2: 400 collected, 450 to collect


DAT:  15%|█████████▌                                                    | 69/450 [01:23<06:43,  1.06s/it]

### Backfill pickles without `parsed` and `score` fields

In [1]:
import os, pickle
import automated_intelligence_tests as ait
from IPython.display import clear_output

def list_pickle_fps(root):
    fps = []
    for dp, _, fns in os.walk(root):
        for n in fns:
            if n.endswith(".pickle") and not n.endswith(".pickle.tmp"):
                fps.append(os.path.join(dp, n))
    return fps

def dump(p, row):
    tmp = p + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(row, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)

def load(p):
    with open(p, "rb") as f:
        return pickle.load(f)

fps = list_pickle_fps("./data/dat/")
n = len(fps)
for i, p in enumerate(fps, 1):
    row = load(p)
    if isinstance(row.get("score"), (int, float)):
        clear_output(wait=True)
        print(f"{i}/{n}  skip score={row['score']}")
        continue
    raw = row.get("raw")
    try:
        parsed = ait.parse("dat", raw, stim=row.get("kwargs")) if raw else None
        score = ait.evaluate("dat", parsed).get("score") if parsed else None
    except Exception:
        parsed, score = None, None
    row["parsed"] = parsed
    row["score"] = score
    dump(p, row)
    raw_show = " ".join(str(raw or "1").split())[:120]
    clear_output(wait=True)
    print(f"{i}/{n}  score={score}  raw={raw_show}")

10193/10193  score=74.14920656453997  raw=dog, happiness, ocean, chair, freedom, apple, time, love, mountain, money
